# Assignment 5: Augmented RAG Learner Guide Assistant

**EDAB6808 · Google Colab · Combined notebook**

This version combines the two notebooks' useful features: BGE embeddings and BM25/dense rank fusion, original-text token chunking and same-page expansion, cross-encoder reranking, source-copying answers, Drive persistence, and a chatbot with evidence and evaluation panels.

The **EDAB6808 Learner Guide is the only chatbot knowledge source**. The assignment brief and method documentation explain the implementation and are not indexed.

Select a GPU runtime, run the cells in order, and upload the guide when prompted. The app uses Qwen2.5-1.5B-Instruct by default. Complete the 18-case evaluation and its quality reviews before reporting final accuracy.

**Execution status:** All outputs are cleared because the models and retrieval design changed. No completed Colab results, accuracy claims or measured speed improvements are supplied. Source copying verifies wording and reference mapping; human review checks meaning and completeness.


## 1. Install the libraries

Install into the current Colab runtime. Colab's existing PyTorch installation is used.
The model downloads happen later. The optional RAGAS section has its own installation switch.


In [ ]:
%pip install -q pymupdf sentence-transformers qdrant-client rank_bm25 transformers accelerate pandas gradio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.5/406.5 kB 20.7 MB/s eta 0:00:00


## 2. Check the Colab environment

Select a GPU runtime for the demonstration. The code also supports CPU, although generation will take longer. Record the device and package versions with the results.


In [ ]:
import sys
import torch
import pymupdf
import pandas as pd
import gradio as gr
from sentence_transformers import SentenceTransformer, CrossEncoder
from qdrant_client import QdrantClient
from transformers import AutoTokenizer, AutoModelForCausalLM
from importlib.metadata import version
from pathlib import Path

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Environment ready.")
print("Python:", sys.version.split()[0])
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if device == "cuda" else "CPU")
print("PyTorch CUDA version:", torch.version.cuda)

for package in [
    "torch", "pymupdf", "sentence-transformers", "qdrant-client",
    "transformers", "accelerate", "rank-bm25", "pandas", "gradio"
]:
    print(f"{package}: {version(package)}")

if device == "cpu":
    print("CPU selected. To use CUDA, choose a GPU runtime in Colab and rerun.")


Environment ready.
Python: 3.13.15
CUDA available: True
Device: Tesla T4
PyTorch CUDA version: 13.0
torch: 2.11.0+cu130
pymupdf: 1.28.2
sentence-transformers: 5.7.0
qdrant-client: 1.19.1
transformers: 5.18.0
accelerate: 1.15.0
rank-bm25: 0.2.2
pandas: 2.2.3
gradio: 6.29.0


## 3. Mount Drive, upload the guide, and set paths

The project folder is `MyDrive/EDAB6808_RAG_Augmented`. Upload **only the EDAB6808 Learner Guide** when prompted.
A cover check rejects a different document. The original PDF, extraction, chunks and results are retained.
Qdrant works on Colab's local disk and is archived to Drive after indexing, avoiding a live database on the Drive mount.
The configuration above is the single place to change models and retrieval settings. Changing settings requires a fresh evaluation.


In [ ]:
import hashlib
import json
import os
import shutil
import tempfile
import zipfile
from time import perf_counter
from IPython.display import display, Markdown
from google.colab import drive, files

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/EDAB6808_RAG_Augmented")
LOCAL_DIR = Path("/content/edab6808_rag_augmented")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_DIR.mkdir(parents=True, exist_ok=True)
PDF_PATH = PROJECT_DIR / "Study_Guide.pdf"
DATA_DIR = PROJECT_DIR / "rag_data"
QDRANT_PATH = LOCAL_DIR / "qdrant_db"
QDRANT_BACKUP = PROJECT_DIR / "qdrant_db.zip"
COLLECTION_NAME = "edab6808_guide_augmented"
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
GENERATION_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
# A 3B Qwen can be tried on a GPU by changing this setting and rerunning.
CHUNK_SIZE, OVERLAP = 220, 40       # BGE tokens, not word estimates
CANDIDATE_K, BM25_K, FUSE_K = 15, 15, 15
TOP_K, RRF_K = 4, 60
RETRIEVAL_MODE = "hybrid"
RETRIEVAL_MODES = ["dense", "dense_rerank", "hybrid"]
PIPELINE_VERSION = "2026-10-09-augmented-colab-v2"

def save_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False),
        encoding="utf-8",
    )
    temporary.replace(path)

def restore_qdrant_archive(destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(QDRANT_BACKUP) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if (
                not target.is_relative_to(destination)
                or not member.filename.startswith("qdrant_db/")
            ):
                raise ValueError("The database backup contains an unexpected path.")
        archive.extractall(destination)

if not PDF_PATH.is_file():
    print("Upload the EDAB6808 Learner Guide PDF.")
    uploaded = files.upload()
    pdf_files = [name for name in uploaded if name.lower().endswith(".pdf")]
    if len(uploaded) != 1 or len(pdf_files) != 1:
        raise ValueError("Upload exactly one learner-guide PDF and rerun this cell.")
    pdf_bytes = uploaded[pdf_files[0]]
    with pymupdf.open(stream=pdf_bytes, filetype="pdf") as document:
        cover = " ".join(document[0].get_text().lower().split())
    if (
        "edab6808" not in cover.replace(" ", "")
        or "data analytics for business" not in cover
    ):
        raise ValueError("The uploaded PDF is not the EDAB6808 learner guide.")
    PDF_PATH.write_bytes(pdf_bytes)
    print("Learner guide saved to the project folder.")

if QDRANT_BACKUP.is_file() and not QDRANT_PATH.exists():
    restore_qdrant_archive(LOCAL_DIR)
    print("Qdrant database restored from its Drive backup.")

def backup_qdrant_to_drive():
    global client
    # Close Qdrant before archiving its files, then reopen for later questions.
    client.close()
    try:
        archive = Path(shutil.make_archive(
            str(LOCAL_DIR / "qdrant_backup"), "zip",
            root_dir=str(LOCAL_DIR), base_dir="qdrant_db",
        ))
        temporary = QDRANT_BACKUP.with_name("qdrant_db.tmp.zip")
        shutil.copyfile(archive, temporary)
        temporary.replace(QDRANT_BACKUP)
    finally:
        client = QdrantClient(path=str(QDRANT_PATH))
    print("Qdrant backup saved:", QDRANT_BACKUP)

print("Study guide:", PDF_PATH)
print("Database working folder:", QDRANT_PATH)
print("Saved results:", DATA_DIR)
print("Retrieval:", RETRIEVAL_MODE, "| Dense/BM25 candidates:", CANDIDATE_K, BM25_K, "| Final passages:", TOP_K)


Mounted at /content/drive
Qdrant database restored from its Drive backup.
Study guide: /content/drive/MyDrive/EDAB6808_RAG_Augmented/Study_Guide.pdf
Database working folder: /content/edab6808_rag_augmented/qdrant_db
Saved results: /content/drive/MyDrive/EDAB6808_RAG_Augmented/rag_data
Retrieval: hybrid | Dense/BM25 candidates: 15 15 | Final passages: 4


## 4. Extract the study guide

Retain the original PDF and uncleaned page text. Physical PDF pages start at 1; the cover has no printed page number. No textbook or other document is indexed.


In [ ]:
if not PDF_PATH.is_file():
    raise FileNotFoundError(
        f"Study guide not found: {PDF_PATH}. Update PDF_PATH in the paths cell."
    )

pdf_name = PDF_PATH.name

with pymupdf.open(PDF_PATH) as pdf:
    raw_pages = [
        {"pdf_page": number + 1, "text": page.get_text("text", sort=True)}
        for number, page in enumerate(pdf)
    ]

cover = " ".join(raw_pages[0]["text"].lower().split())
if "edab6808" not in cover.replace(" ", "") or "data analytics for business" not in cover:
    raise ValueError("This PDF does not match the supplied EDAB6808 study guide.")

print("File:", pdf_name)
print("PDF pages:", len(raw_pages))
print("Pages containing text:", sum(bool(page["text"].strip()) for page in raw_pages))


File: Study_Guide.pdf
PDF pages: 16
Pages containing text: 16


## 5. Inspect the extraction

Inspect instructor information and the assessment table before indexing. Table labels and values can be arranged awkwardly by PDF extraction.


In [ ]:
for pdf_page in [3, 15]:
    print(f"\nPDF page {pdf_page}")
    print(raw_pages[pdf_page - 1]["text"])



PDF page 3
                   1. INSTRUCTOR INFORMATION


Dear student/ colleague,

Welcome to EDAB6808, Introduction to Data Analytics for Business! I am a data scientist,

and I will be responsible for teaching this module. My name is Dr Herkulaas MvE Combrink,

and I have extensive experience in complex disease modelling, business analytics, data science,

statistics, prediction modelling, exploratory data analysis, and complex data analysis – all

within simple and complex systems. Here is all the information that you will need to contact

us:

           LECTURER

 Lecturer:        Herkulaas MvE Combrink/ V Oelofse / J Trip

 Office number: FGG 352

 Email address:  CombrinkHM@UFS.ac.za; OelofseVEO@ufs.ac.za; TripJK@ufs.ac.za

 Phone number:  051 401 7408

 Consulting     To book an appointment, use the template below:

 hours:          Student number as subject title and 1 (one) word to summarise the

                nature of the email. For example: 2020202011 – Inquiry Then ty

## 6. Light cleaning

Remove the guide’s repeated page footers and excess blank lines. Retain raw text and preserve substantive source wording.


In [ ]:
import re

clean_pages = []
removed_footers = 0

for page in raw_pages:
    text, count = re.subn(
        r"(?im)^[ \t]*\d+[ \t]*\|[ \t]*p[ \t]*a[ \t]*g[ \t]*e[ \t]*$",
        "",
        page["text"]
    )

    text = re.sub(r"\n[ \t]*\n(?:[ \t]*\n)+", "\n\n", text).strip()

    clean_pages.append({**page, "text": text})
    removed_footers += count

print("Pages retained:", len(clean_pages))
print("Footers removed:", removed_footers)

# Inspect the cleaned assessment page
print(clean_pages[14]["text"])


Pages retained: 16
Footers removed: 15
a) The entire module mark (100%), will be based on the continuous assessment of the

    module. If a student misses an assessment opportunity without a valid reason, then the

    student will receive an incomplete for the module. A valid excuse is stipulated by the

    faculty rules, and determined by the UFS policy, not the lecturer. Therefore, rule

    implementation will be on the basis of the UFS rules, and no amount of persuasion will

    convince the lecturer of a different outcome, unless stipulated by the UFS policy. All

    people are created equal, and a fair judgement within the system will be implemented

    for all people, using evidence, and a contextual understanding of the rules and its

    limitations.

b)  If an excuse has been found valid, the student will have an opportunity to follow-up

   and defend the assignment in an oral test. The scope of the oral test will be the same as

    the assessment, except all the ques

## 7. Chunking and source metadata

Use **220 BGE tokens with 40 tokens of overlap**, bounded by PDF page. Token offsets preserve original text.
Main numbered headings and explicit `Unit n:` headings provide metadata. Bold body text is not treated as a new section.
A chunk spanning a heading records both sections. Stable IDs, printed/PDF pages and character positions support inspection and citations.
Smaller chunks can isolate facts; larger chunks retain surrounding explanations and table context. Expansion later adds the following chunk when it is on the same page.


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL, use_fast=True)
assert 0 <= OVERLAP < CHUNK_SIZE
heading_pattern = r"(?m)^[ \t]*([1-9]\.[ \t]+[A-Z][A-Z &()/\-]+)[ \t]*$"
unit_pattern = r"(?m)^[ \t]*(?:[•●▪◦\-][ \t]*)?(Unit[ \t]+\d+:[^\n]+)[ \t]*$"
chunks, page_token_offsets = [], {}
current_heading, current_unit = "Cover", None

for page in clean_pages:
    text, pdf_page = page["text"], page["pdf_page"]
    if pdf_page == 2:
        current_heading, current_unit = "Table of Contents", None
    events = [(m.start(), "main", m.group(1).strip())
              for m in re.finditer(heading_pattern, text)]
    events += [(m.start(), "unit", m.group(1).strip())
               for m in re.finditer(unit_pattern, text)]
    events.sort(key=lambda event: event[0])
    positions = [(0, current_heading, current_unit)]
    for position, kind, label in events:
        if kind == "main":
            current_heading, current_unit = label, None
        else:
            current_unit = label
        positions.append((position, current_heading, current_unit))

    offsets = tokenizer(
        text, add_special_tokens=False, truncation=False,
        return_offsets_mapping=True, verbose=False,
    )["offset_mapping"]
    page_token_offsets[pdf_page] = offsets
    for number, start in enumerate(range(0, len(offsets), CHUNK_SIZE - OVERLAP), 1):
        end = min(start + CHUNK_SIZE, len(offsets))
        char_start, char_end = offsets[start][0], offsets[end - 1][1]
        initial = next(item for item in reversed(positions) if item[0] <= char_start)
        applicable = [initial] + [item for item in positions if char_start < item[0] < char_end]
        labels = list(dict.fromkeys(item[1] for item in applicable))
        units = list(dict.fromkeys(item[2] for item in applicable if item[2]))
        chunks.append({
            "chunk_id": f"edab_p{pdf_page:02d}_c{number:02d}",
            "text": text[char_start:char_end],
            "pdf_page": pdf_page,
            "printed_page": pdf_page - 1 if pdf_page > 1 else None,
            "section": " / ".join(labels),
            "unit": " / ".join(units) or None,
            "char_start": char_start, "char_end": char_end,
        })
        if end == len(offsets):
            break

print("Chunks:", len(chunks), "| PDF pages:", len({c["pdf_page"] for c in chunks}))
display(pd.DataFrame(chunks)[["chunk_id", "printed_page", "section", "unit"]].head(12))


config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Chunks: 38 | PDF pages: 16


,chunk_id,printed_page,section,unit
0,edab_p01_c01,NaN,Cover,None
1,edab_p02_c01,1.0,Table of Contents,None
2,edab_p02_c02,1.0,Table of Contents,None
3,edab_p02_c03,1.0,Table of Contents,None
4,edab_p02_c04,1.0,Table of Contents,None
5,edab_p02_c05,1.0,Table of Contents,None
6,edab_p03_c01,2.0,1. INSTRUCTOR INFORMATION,None
7,edab_p03_c02,2.0,1. INSTRUCTOR INFORMATION,None
8,edab_p04_c01,3.0,2. MODULE DESCRIPTION,None
9,edab_p04_c02,3.0,2. MODULE DESCRIPTION,None


## 8. Check coverage and inspect the chunk-size trade-off

Check every non-whitespace character, unique IDs, token lengths and assessment metadata. The extra table counts alternative page-bounded token windows without rebuilding the index.


In [ ]:
# Confirm that every non-whitespace character is covered by a chunk.
for page in clean_pages:
    covered = [False] * len(page["text"])
    for chunk in chunks:
        if chunk["pdf_page"] == page["pdf_page"]:
            for i in range(chunk["char_start"], chunk["char_end"]):
                covered[i] = True
    assert all(covered[i] or character.isspace()
               for i, character in enumerate(page["text"])), page["pdf_page"]

assert len({c["chunk_id"] for c in chunks}) == len(chunks)

max_tokens = max(len(tokenizer(c["text"])["input_ids"]) for c in chunks)
assert max_tokens <= min(512, tokenizer.model_max_length), "A chunk exceeds the BGE input limit."

print("Text coverage: complete")
print("Chunk IDs: unique")
print("Largest chunk including special tokens:", max_tokens)

example = next(c for c in chunks
               if "Assignment 5" in c["text"] and "ASSESSMENT" in c["section"])
print("\nExample:", example["chunk_id"])
print("PDF page:", example["pdf_page"], "| Printed page:", example["printed_page"])
print("Section:", example["section"])
print(example["text"])


Text coverage: complete
Chunk IDs: unique
Largest chunk including special tokens: 222

Example: edab_p15_c02
PDF page: 15 | Printed page: 14
Section: 7. ASSESSMENT OF LEARNING
assessment, except all the questions will be predetermined by the lecturer and a

    designated time will be given for the oral tests, depending on the number of students.

   None of the questions for an oral test will be repeated, and all the oral tests will occur

     at the same time. Feedback will be given of the oral test immediately after the oral, as

    the assessment of the answers will be based on a rubric.

ASSESSMENT             TYPE (E.G. WRITTEN/ORAL/PRACTICAL)             CONTRIBUTION
                                                                     TO MODULE

                                                             MARK (weight)

 Assignment 1         LookerStudio + Notebook + PowerPoint presentation 30 minutes per        20%

                                student (20 minutes presenta

In [ ]:
import math
tradeoff_rows = []
for size in [140, 220, 300, 420]:
    count = sum(
        0 if not offsets else 1 + max(0, math.ceil((len(offsets) - size) / (size - OVERLAP)))
        for offsets in page_token_offsets.values()
    )
    tradeoff_rows.append({
        "Chunk tokens": size, "Overlap tokens": OVERLAP,
        "Page-bounded chunks": count,
        "Maximum raw evidence tokens": TOP_K * size,
    })
display(pd.DataFrame(tradeoff_rows))
print("Counts describe chunking choices; they are not measured retrieval or accuracy results.")


,Chunk tokens,Overlap tokens,Page-bounded chunks,Maximum raw evidence tokens
0,140,40,62,560
1,220,40,38,880
2,300,40,27,1200
3,420,40,22,1680


Counts describe chunking choices; they are not measured retrieval or accuracy results.


## 9. Create BGE embeddings

`BAAI/bge-small-en-v1.5` is an open-weight SentenceTransformers model with **384-dimensional** vectors.
Document chunks are embedded as written. Queries receive BGE's retrieval prefix. Normalize vectors and use cosine similarity.


In [ ]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL, device=device)
assert max_tokens <= embedding_model.max_seq_length
assert embedding_model.get_sentence_embedding_dimension() == 384
started = perf_counter()
embeddings = embedding_model.encode(
    [chunk["text"] for chunk in chunks], batch_size=32,
    normalize_embeddings=True, show_progress_bar=True,
)
assert embeddings.shape == (len(chunks), 384)
print("Embedding model:", EMBEDDING_MODEL)
print("Shape:", embeddings.shape, "| Seconds:", round(perf_counter() - started, 2))


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_1287/984790800.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  assert embedding_model.get_sentence_embedding_dimension() == 384


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embedding model: BAAI/bge-small-en-v1.5
Shape: (38, 384) | Seconds: 2.8


## 10. Store vectors and metadata in Qdrant Local

Rebuild only the augmented assignment collection so reruns do not duplicate points. Count vectors, save source and index manifests, then close/archive/reopen Qdrant for a consistent Drive backup.


In [ ]:
from qdrant_client import QdrantClient, models

assert len(chunks) == embeddings.shape[0]
DATA_DIR.mkdir(parents=True, exist_ok=True)
source_sha256 = hashlib.sha256(PDF_PATH.read_bytes()).hexdigest()

if "demo" in globals():
    demo.close()
if "client" in globals():
    client.close()
client = QdrantClient(path=str(QDRANT_PATH))

if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=models.VectorParams(
        size=int(embeddings.shape[1]), distance=models.Distance.COSINE
    ),
)

points = [
    models.PointStruct(
        id=i,
        vector=embeddings[i].tolist(),
        payload={
            **chunk,
            "source_pdf": pdf_name,
            "source_sha256": source_sha256,
        },
    )
    for i, chunk in enumerate(chunks)
]
client.upsert(collection_name=COLLECTION_NAME, points=points, wait=True)
stored_count = client.count(COLLECTION_NAME, exact=True).count
assert stored_count == len(chunks), "Stored vector count does not match."

save_json(DATA_DIR / "raw_pages.json", raw_pages)
save_json(DATA_DIR / "clean_pages.json", clean_pages)
save_json(DATA_DIR / "chunks.json", chunks)
save_json(DATA_DIR / "index_manifest.json", {
    "source_pdf": pdf_name,
    "source_sha256": source_sha256,
    "embedding_model": EMBEDDING_MODEL, "query_prefix": QUERY_PREFIX,
    "dimension": int(embeddings.shape[1]),
    "chunk_size": CHUNK_SIZE,
    "overlap": OVERLAP,
    "collection": COLLECTION_NAME,
    "chunk_count": len(chunks),
})

print("Collection:", COLLECTION_NAME)
print("Chunks:", len(chunks), "| Stored vectors:", stored_count)
print("Database:", QDRANT_PATH)

backup_qdrant_to_drive()


Collection: edab6808_guide_augmented
Chunks: 38 | Stored vectors: 38
Database: /content/edab6808_rag_augmented/qdrant_db
Qdrant backup saved: /content/drive/MyDrive/EDAB6808_RAG_Augmented/qdrant_db.zip


## 11. Dense retrieval, BM25, rank fusion, expansion and reranking

Three methods use the same final **k = 4**:

| Method | Evidence selection |
|---|---|
| `dense` | Raw dense top four |
| `dense_rerank` | Dense top 15, same-page expansion, cross-encoder reranking |
| `hybrid` | Dense top 15 + BM25 top 15, reciprocal rank fusion, top 15 expanded candidates, reranking |

BM25 preserves lexical matches such as assignment numbers and tool names. RRF combines rankings without mixing incompatible scores.
Expansion retains neighbouring source context. Four final passages allow multi-section evidence while limiting the prompt.
Rankings and shared words explain retrieval, but cannot establish answerability. The Qwen evidence gate makes that decision.


In [ ]:
import numpy as np
from collections import defaultdict
from rank_bm25 import BM25Okapi

RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L6-v2"
reranker = CrossEncoder(RERANKER_MODEL, device=device, max_length=512)
STOP_WORDS = set("a an the of to in on for and or is are was were be been it this that with as by at from what which who how when where why do does can should would".split())

def lexical_tokens(text):
    return [word for word in re.findall(r"[a-z0-9]+", text.lower()) if word not in STOP_WORDS]

chunk_positions = {chunk["chunk_id"]: i for i, chunk in enumerate(chunks)}
chunk_lookup = {chunk["chunk_id"]: chunk for chunk in chunks}
page_texts = {page["pdf_page"]: page["text"] for page in clean_pages}
bm25 = BM25Okapi([lexical_tokens(chunk["text"]) for chunk in chunks])

def retrieve_dense(question, k=CANDIDATE_K):
    query_vector = embedding_model.encode(
        QUERY_PREFIX + question, normalize_embeddings=True,
    ).tolist()
    hits = client.query_points(
        collection_name=COLLECTION_NAME, query=query_vector,
        limit=k, with_payload=True,
    ).points
    return [{**hit.payload, "dense_score": float(hit.score), "dense_rank": rank,
             "chunk_ids": [hit.payload["chunk_id"]]}
            for rank, hit in enumerate(hits, 1)]

def retrieve_bm25(question, k=BM25_K):
    scores = bm25.get_scores(lexical_tokens(question))
    order = np.argsort(-scores, kind="stable")[:k]
    return [(chunks[i]["chunk_id"], float(scores[i])) for i in order if scores[i] > 0]

def reciprocal_rank_fusion(rankings, constant=RRF_K):
    scores = defaultdict(float)
    for ranking in rankings:
        for rank, (chunk_id, _) in enumerate(ranking, 1):
            scores[chunk_id] += 1.0 / (constant + rank)
    return sorted(scores.items(), key=lambda item: (-item[1], item[0]))

def expand_context(chunk):
    # Return a copy: expanding evidence must not mutate the index or other calls.
    result = {**chunk, "chunk_ids": list(chunk.get("chunk_ids", [chunk["chunk_id"]]))}
    position = chunk_positions[chunk["chunk_id"]]
    if position + 1 < len(chunks):
        following = chunks[position + 1]
        if following["pdf_page"] == chunk["pdf_page"]:
            result["char_end"] = max(chunk["char_end"], following["char_end"])
            result["text"] = page_texts[chunk["pdf_page"]][chunk["char_start"]:result["char_end"]]
            result["chunk_ids"] = list(dict.fromkeys(result["chunk_ids"] + [following["chunk_id"]]))
            labels = chunk["section"].split(" / ") + following["section"].split(" / ")
            result["section"] = " / ".join(dict.fromkeys(labels))
            units = [unit for part in [chunk.get("unit"), following.get("unit")] if part
                     for unit in part.split(" / ")]
            result["unit"] = " / ".join(dict.fromkeys(units)) or None
    return result

def retrieve(question, top_k=TOP_K, mode=RETRIEVAL_MODE):
    if mode not in RETRIEVAL_MODES:
        raise ValueError("Choose dense, dense_rerank or hybrid retrieval.")
    dense = retrieve_dense(question, k=top_k if mode == "dense" else CANDIDATE_K)
    if mode == "dense":
        return dense[:top_k]
    dense_lookup = {chunk["chunk_id"]: chunk for chunk in dense}
    if mode == "hybrid":
        lexical = retrieve_bm25(question)
        lexical_lookup = {cid: (rank, score) for rank, (cid, score) in enumerate(lexical, 1)}
        fused = reciprocal_rank_fusion([
            [(chunk["chunk_id"], chunk["dense_score"]) for chunk in dense], lexical,
        ])[:FUSE_K]
        candidates = []
        for cid, fusion_score in fused:
            original = dense_lookup.get(cid, {
                **chunk_lookup[cid], "source_pdf": pdf_name, "source_sha256": source_sha256,
                "dense_rank": None, "dense_score": None, "chunk_ids": [cid],
            })
            rank, score = lexical_lookup.get(cid, (None, None))
            candidates.append({**original, "bm25_rank": rank, "bm25_score": score,
                               "fusion_score": float(fusion_score)})
    else:
        candidates = dense
    candidates = [expand_context(chunk) for chunk in candidates]
    if not candidates:
        return []
    scores = reranker.predict([(question, chunk["text"]) for chunk in candidates], show_progress_bar=False)
    for chunk, score in zip(candidates, scores):
        chunk["rerank_score"] = float(score)
    return sorted(candidates, key=lambda chunk: (-chunk["rerank_score"], chunk["chunk_id"]))[:top_k]

def why_relevant(question, chunk):
    shared = sorted(set(lexical_tokens(question)) & set(lexical_tokens(chunk["text"])))
    pieces = [f"Dense rank: {chunk.get('dense_rank')}"]
    if chunk.get("bm25_rank") is not None:
        pieces.append(f"BM25 rank: {chunk['bm25_rank']}")
    if chunk.get("rerank_score") is not None:
        pieces.append(f"Reranker: {chunk['rerank_score']:.3f}")
    pieces.append("Shared words: " + (", ".join(shared) or "none"))
    return " | ".join(pieces)

print("Retrieval ready:", RETRIEVAL_MODE, "| Final k:", TOP_K)


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Retrieval ready: hybrid | Final k: 4


## 12. Inspect the leading assessment passage

Inspect evidence before generation. Check that the Assignment 5 row retains its 20% contribution and assessment section metadata.


In [ ]:
question = "How much does Assignment 5 contribute to the module mark?"
retrieved_chunks = retrieve(question)
for rank, chunk in enumerate(retrieved_chunks, 1):
    print(f"\n[{rank}] {chunk['chunk_id']} | Guide page {chunk['printed_page']} | {chunk['section']}")
    print(why_relevant(question, chunk))
    print(chunk["text"])



[1] edab_p15_c01 | Guide page 14 | 7. ASSESSMENT OF LEARNING
Dense rank: 7 | BM25 rank: 3 | Reranker: 1.870 | Shared words: 5, assignment, mark, module
a) The entire module mark (100%), will be based on the continuous assessment of the

    module. If a student misses an assessment opportunity without a valid reason, then the

    student will receive an incomplete for the module. A valid excuse is stipulated by the

    faculty rules, and determined by the UFS policy, not the lecturer. Therefore, rule

    implementation will be on the basis of the UFS rules, and no amount of persuasion will

    convince the lecturer of a different outcome, unless stipulated by the UFS policy. All

    people are created equal, and a fair judgement within the system will be implemented

    for all people, using evidence, and a contextual understanding of the rules and its

    limitations.

b)  If an excuse has been found valid, the student will have an opportunity to follow-up

   and defend the a

## 13. Check retrieval across guide sections

These source-presence checks inspect evidence. They do not grade generated answers.


In [ ]:
import re
import pandas as pd

checks = [
    (
        "What office number is listed for the module lecturers?",
        2,
        r"\bFGG 352\b"
    ),
    (
        "How many credits and notional learning hours does EDAB6808 require?",
        5,
        r"\b32-credit\b.*\b320 notional learning hours\b"
    ),
    (
        "Is there a prescribed textbook for EDAB6808?",
        13,
        r"no prescribed textbook"
    ),
    (
        "What percentage of the sessions must students attend?",
        15,
        r"attend 100% of the sessions"
    )
]

check_results = []

for question, expected_page, pattern in checks:
    results = retrieve(question)
    evidence_rank = None

    for rank, chunk in enumerate(results, start=1):
        text = " ".join(chunk["text"].split())

        if (
            chunk["printed_page"] == expected_page
            and re.search(pattern, text, flags=re.IGNORECASE)
        ):
            evidence_rank = rank
            break

    check_results.append({
        "Question": question,
        "Expected printed page": expected_page,
        "Evidence rank": evidence_rank if evidence_rank else "—",
        "Evidence found": evidence_rank is not None
    })

display(pd.DataFrame(check_results))


,Question,Expected printed page,Evidence rank,Evidence found
0,What office number is listed for the module le...,2,1,True
1,How many credits and notional learning hours d...,5,1,True
2,Is there a prescribed textbook for EDAB6808?,13,2,True
3,What percentage of the sessions must students ...,15,1,True


## 14. Load the local response model

Load the configured instruction-tuned Qwen once. Default **Qwen2.5-1.5B-Instruct** keeps the Colab demonstration manageable.
CUDA uses float16 and CPU uses float32. The supplied required-stack table calls for a local instruction-tuned Qwen/Gemma;
this notebook uses that model without additional project-specific training. A larger model should be judged on a new evaluation.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

llm_dtype = torch.float16 if device == "cuda" else torch.float32

llm_tokenizer = AutoTokenizer.from_pretrained(GENERATION_MODEL)

llm = AutoModelForCausalLM.from_pretrained(
    GENERATION_MODEL,
    dtype=llm_dtype,
    device_map=device
)

llm.eval()

print("Model:", GENERATION_MODEL)
print("Device:", llm.device)
print("Model loaded successfully.")


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model: Qwen/Qwen2.5-1.5B-Instruct
Device: cuda:0
Model loaded successfully.


## 15. Check answerability

Qwen checks whether the retrieved excerpts answer every part of the question. A relevance score cannot supply a missing fact such as a deadline. Invalid model output remains a pipeline error, distinct from a genuine refusal.


In [ ]:
def check_answerability(question, evidence, show_details=False):
    context = "\n\n".join(
        f"[{i}] {chunk['text']}"
        for i, chunk in enumerate(evidence, start=1)
    )

    messages = [
        {
            "role": "system",
            "content": (
                "Read the excerpts and form a short answer to the question. "
                "Use only information in the excerpts. "
                "Combine facts from different excerpts when necessary. "
                "Read table values together with their headings. "
                "A supported negative answer, such as something being absent "
                "or not required, still counts as an answer. "
                "Choose YES when the excerpts answer every part of the question. "
                "Choose NO when a requested fact is missing. "
                "Match the information requested: a presentation duration "
                "cannot supply a submission date, and an assessment weight "
                "cannot supply a monetary fee. "
                "Reply using exactly two lines:\n"
                "Evidence: <a brief supported answer, or NONE>\n"
                "Decision: YES or NO"
            )
        },
        {
            "role": "user",
            "content": f"Excerpts:\n{context}\n\nQuestion: {question}"
        }
    ]

    for attempt in range(2):
        inputs = llm_tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt"
        ).to(llm.device)

        with torch.inference_mode():
            outputs = llm.generate(
                **inputs,
                max_new_tokens=160 if attempt == 0 else 8,
                do_sample=False,
                temperature=None,
                top_p=None,
                top_k=None,
                pad_token_id=llm_tokenizer.eos_token_id
            )

        new_tokens = outputs[0, inputs["input_ids"].shape[1]:]
        response = llm_tokenizer.decode(
            new_tokens,
            skip_special_tokens=True
        ).strip()

        if show_details:
            print(f"Attempt {attempt + 1}:\n{response}")

        normalized = response.upper().rstrip(".!")

        if normalized in {"YES", "NO"}:
            return normalized

        if normalized in {"NONE", "EVIDENCE: NONE"}:
            return "NO"

        match = re.search(
            r"(?im)^[ \t]*Decision:[ \t]*(YES|NO)[ \t]*[.!]?[ \t]*$",
            response
        )

        if match:
            return match.group(1).upper()

        if attempt == 0:
            messages.extend([
                {"role": "assistant", "content": response},
                {
                    "role": "user",
                    "content": (
                        "Give only the decision word: YES or NO. "
                        "Choose YES if the excerpts answer every part "
                        "of the question. Otherwise choose NO."
                    )
                }
            ])

    return "UNKNOWN"


## 16. Inspect answerability decisions

Run these diagnostic examples with the current retrieval and model. The complete evaluation comes later.


In [ ]:
answerability_cases = [
    ("How much does Assignment 5 contribute to the module mark?", "YES"),
    ("What is the exact submission deadline for Assignment 5?", "NO"),
    ("What is the tuition fee for EDAB6808?", "NO"),
]

answerability_results = []

for question, expected in answerability_cases:
    print("\nQuestion:", question)

    decision = check_answerability(
        question, retrieve(question), show_details=True
    )

    answerability_results.append({
        "Question": question,
        "Expected": expected,
        "Model decision": decision,
        "Passed": decision == expected,
    })

display(pd.DataFrame(answerability_results))



Question: How much does Assignment 5 contribute to the module mark?
Attempt 1:
YES

Question: What is the exact submission deadline for Assignment 5?
Attempt 1:
NONE

Question: What is the tuition fee for EDAB6808?
Attempt 1:
NONE


,Question,Expected,Model decision,Passed
0,How much does Assignment 5 contribute to the m...,YES,YES,True
1,What is the exact submission deadline for Assi...,NO,NO,True
2,What is the tuition fee for EDAB6808?,NO,NO,True


## 17. Generate source-grounded answers and metadata citations

Qwen selects labelled source lines. If it returns source wording instead, match that wording to the shortest contiguous source-line group. Python groups adjacent lines, verifies exact source spans and constructs references from metadata.
This prevents invented wording and page numbers. Exact wording can still be irrelevant or incomplete, so human review grades meaning.
Unsupported questions receive the fixed refusal. Formatting failures are recorded as errors and can be retried with another question.


In [ ]:
import re
import torch

ABSTAIN = (
    "The study guide does not provide enough information "
    "to answer this question."
)


def source_text_to_labels(response, line_lookup):
    # Recover copied wording, then let the usual checker copy ORIGINAL lines.
    text = response.strip()
    text = re.sub(r"(?i)^(?:Answer|Evidence|Response):\s*", "", text)
    text = text.rstrip(".").strip().strip('"“”')
    target = " ".join(text.split()).casefold()
    if not target:
        raise ValueError("The model selected no source text.")
    pattern = r"(?<!\w)" + re.escape(target) + r"(?!\w)"
    numbers = sorted(line_lookup)
    matches = []
    for start, first in enumerate(numbers):
        reference = line_lookup[first][0]
        chosen, pieces = [], []
        previous = first - 1
        for number in numbers[start:]:
            ref, line = line_lookup[number]
            if ref != reference or number != previous + 1:
                break
            chosen.append(number)
            pieces.append(line)
            span = " ".join(" ".join(pieces).split())
            if re.search(pattern, span.casefold()):
                matches.append((len(span), len(chosen), first, chosen.copy()))
                break
            previous = number
    if not matches:
        raise ValueError("The model response does not match the retrieved source wording.")
    selected = min(matches)[3]
    return ", ".join(f"L{number}" for number in selected)


def copy_selected_lines(response, line_lookup, evidence):
    # Accept line labels or copied wording that can be mapped to source lines.
    response = response.strip()
    if response.startswith("[") and response.endswith("]"):
        response = response[1:-1].replace('"', "").replace("'", "")
    if not re.fullmatch(
        r"L\d+(?:[\s,]+L\d+)*", response.strip()
    ):
        response = source_text_to_labels(response, line_lookup)

    selected = sorted({
        int(number)
        for number in re.findall(r"L(\d+)", response)
    })

    if any(number not in line_lookup for number in selected):
        raise ValueError("An unknown source-line label was selected.")

    # Combine neighbouring lines from the same excerpt.
    groups = []

    for number in selected:
        reference, text = line_lookup[number]

        if (
            groups
            and groups[-1]["reference"] == reference
            and groups[-1]["last"] == number - 1
        ):
            groups[-1]["lines"].append(text)
            groups[-1]["last"] = number
        else:
            groups.append({
                "reference": reference,
                "last": number,
                "lines": [text],
            })

    answer_lines = []

    for group in groups:
        passage = " ".join(" ".join(group["lines"]).split())
        reference = group["reference"]
        source_text = " ".join(
            evidence[reference - 1]["text"].split()
        )

        if passage not in source_text:
            raise ValueError(
                "The selected passage is not an exact source span."
            )

        answer_lines.append(f'"{passage}" [{reference}]')

    references = sorted({
        group["reference"] for group in groups
    })

    return {
        "answer": "\n".join(answer_lines),
        "sources": [
            {
                **evidence[reference - 1],
                "reference": f"[{reference}]",
            }
            for reference in references
        ],
        "retrieved_chunks": evidence,
        "selected_line_ids": selected,
        "verified_passages": [
            {"reference": group["reference"],
             "text": " ".join(" ".join(group["lines"]).split())}
            for group in groups
        ],
        "status": "answered",
    }


def answer_question(question, retrieval_mode=RETRIEVAL_MODE, evidence=None):
    if not isinstance(question, str) or not question.strip():
        raise ValueError("Enter a question.")
    question = question.strip()
    if len(question) > 1500:
        raise ValueError("Please shorten the question to 1,500 characters.")
    if evidence is None:
        evidence = retrieve(question, mode=retrieval_mode)

    decision = (
        check_answerability(question, evidence)
        if evidence else "NO"
    )

    if decision == "NO":
        return {
            "answer": ABSTAIN,
            "sources": [],
            "retrieved_chunks": evidence,
            "verified_passages": [],
            "selected_line_ids": [],
            "status": "abstained",
        }

    if decision != "YES":
        raise ValueError(
            "The evidence check returned an invalid decision."
        )

    # Give each non-empty source line a unique label.
    line_lookup = {}
    context_lines = []

    for reference, chunk in enumerate(evidence, start=1):
        context_lines.append(f"\nExcerpt {reference} | Guide page {chunk['printed_page']} | {chunk['section']}:")
        for line in chunk["text"].splitlines():
            if not line.strip():
                continue

            number = len(line_lookup) + 1
            line_lookup[number] = (reference, line.strip())
            context_lines.append(f"L{number}: {line.strip()}")

    context = "\n".join(context_lines)

    messages = [
        {
            "role": "system",
            "content": (
                "Select source lines that answer the question. "
                "Read surrounding lines to understand tables. "
                "Select all lines needed to answer every part. "
                "When a sentence wraps across lines, select the "
                "lines needed to include its complete meaning. "
                "Use the smallest sufficient selection. "
                "Reply only with the selected line labels, "
                "separated by commas. Include no source text "
                "or explanation."
            ),
        },
        {
            "role": "user",
            "content": (
                f"Source lines:\n{context}\n\n"
                f"Question: {question}"
            ),
        },
    ]

    for attempt in range(2):
        inputs = llm_tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
        ).to(llm.device)

        with torch.inference_mode():
            outputs = llm.generate(
                **inputs,
                max_new_tokens=96,
                do_sample=False,
                temperature=None,
                top_p=None,
                top_k=None,
                pad_token_id=llm_tokenizer.eos_token_id,
            )

        new_tokens = outputs[0, inputs["input_ids"].shape[1]:]
        response = llm_tokenizer.decode(
            new_tokens, skip_special_tokens=True
        ).strip()

        try:
            return copy_selected_lines(
                response, line_lookup, evidence
            )
        except ValueError as error:
            last_error = str(error)

        if attempt == 0:
            messages.extend([
                {"role": "assistant", "content": response},
                {
                    "role": "user",
                    "content": (
                        "Return only valid line labels from the "
                        "source, separated by commas. Select the "
                        "lines that answer the question."
                    ),
                },
            ])

    raise ValueError(
        f"{last_error}\nModel output: {response}"
    )



print("One final answer function loaded: source-line selection and verified source-text recovery.")


One final answer function loaded: source-line selection and verified source-text recovery.


## 18. Check citation construction without calling the model

This guard check uses a small fixture to verify source copying and reject invalid labels. It does not measure model accuracy.


In [ ]:
probe_evidence = [{
    "text": "First source line.\nSecond source line.",
    "source_pdf": pdf_name, "pdf_page": 3, "printed_page": 2,
    "section": "1. INSTRUCTOR INFORMATION", "chunk_id": "probe",
}]
probe_lookup = {1: (1, "First source line."),
                2: (1, "Second source line.")}
probe = copy_selected_lines("L1, L2", probe_lookup, probe_evidence)
assert probe["answer"] == '"First source line. Second source line." [1]'
assert probe["sources"][0]["printed_page"] == 2
assert probe["sources"][0]["reference"] == "[1]"
for invalid in ["L999", "L1, invented words", ""]:
    try:
        copy_selected_lines(invalid, probe_lookup, probe_evidence)
    except ValueError:
        pass
    else:
        raise AssertionError("An invalid selection was accepted.")
print("Source copying, citation mapping, and invalid-selection checks passed.")


Source copying, citation mapping, and invalid-selection checks passed.


## 19. Verify the Drive backup in a fresh process

Restore the archive to a separate temporary directory and query it from another Python process. Check stored vectors, source hash and metadata while leaving the active chatbot database open.


In [ ]:
import subprocess

persistence_passed = False
query_vector = embedding_model.encode(
    QUERY_PREFIX + "What is the lecturers' office number?", normalize_embeddings=True
).tolist()
restart_check = """
import json, sys
from qdrant_client import QdrantClient
data = json.load(sys.stdin)
client = QdrantClient(path=data['path'])
try:
    count = client.count(data['collection'], exact=True).count
    saved = client.retrieve(data['collection'], ids=[0], with_vectors=True)[0]
    assert count == data['count']
    assert saved.payload['source_sha256'] == data['source_sha256']
    assert saved.payload['chunk_id'] == data['first_chunk_id']
    assert len(saved.vector) == data['dimension']
    hits = client.query_points(
        collection_name=data['collection'], query=data['query'],
        limit=1, with_payload=True
    ).points
    assert hits and hits[0].payload['source_sha256'] == data['source_sha256']
    print(json.dumps({'vectors': count, 'first_chunk': saved.payload['chunk_id'],
                      'query_chunk': hits[0].payload['chunk_id']}))
finally:
    client.close()
"""

with tempfile.TemporaryDirectory(prefix="qdrant_restore_", dir=str(LOCAL_DIR)) as folder:
    restore_qdrant_archive(folder)
    restored_path = Path(folder) / "qdrant_db"
    try:
        checked = subprocess.run(
            [sys.executable, "-c", restart_check],
            input=json.dumps({
                "path": str(restored_path), "collection": COLLECTION_NAME,
                "count": len(chunks), "source_sha256": source_sha256,
                "first_chunk_id": chunks[0]["chunk_id"],
                "dimension": int(embeddings.shape[1]), "query": query_vector,
            }),
            capture_output=True, text=True, timeout=45, check=True,
        )
        details = json.loads(checked.stdout)
        persistence_passed = True
        print("Restored Drive-backup persistence check passed:", details)
    except subprocess.CalledProcessError as error:
        raise RuntimeError(
            "Drive-backup persistence check failed: " + error.stderr.strip()
        ) from error


Restored Drive-backup persistence check passed: {'vectors': 38, 'first_chunk': 'edab_p01_c01', 'query_chunk': 'edab_p03_c01'}


## 20. Define eighteen evaluation questions

**10 direct, 4 multi-section and 4 unsupported questions** cover six main guide sections.
The test set combines concise factual checks with tools, Bayesian networks, the schedule and near-domain unsupported requests.
Expected answers and source rules are used for evaluation only; they are never passed to Qwen. The guide states three weeks for Unit 3.


In [ ]:
evaluation_df = pd.DataFrame([
    ("Q01", "direct", "What is the lecturers' office number?", "FGG 352", [2], [1]),
    ("Q02", "direct", "What should the email subject contain when booking an appointment?",
     "The student number and one word summarising the enquiry.", [2], [1]),
    ("Q03", "direct", "How many credits is EDAB6808 worth?", "32 credits", [None, 5], [0, 4]),
    ("Q04", "direct", "What is the minimum total number of notional learning hours?", "320 hours", [5], [4]),
    ("Q05", "direct", "Is there a prescribed textbook for this module?", "There is no prescribed textbook.", [13], [6]),
    ("Q06", "direct", "How much does Assignment 5 contribute to the module mark?", "20%", [14], [7]),
    ("Q07", "direct", "How much does Assignment 6 contribute to the module mark?", "10%", [14], [7]),
    ("Q08", "direct", "What percentage of sessions are students expected to attend?", "100%", [15], [8]),
    ("Q09", "direct", "Which Python libraries are named for data cleaning and visualisation?",
     "Pandas, Matplotlib and Seaborn.", [10], [4]),
    ("Q10", "direct", "Which components make up a Bayesian network?",
     "Nodes (variables), edges (dependencies), and conditional probabilities.", [7], [4]),
    ("Q11", "multi_section", "What are the minimum weekly study hours outside class, and what percentage of sessions must students attend?",
     "At least 12 hours outside class per week and 100% attendance.", [5, 15], [4, 8]),
    ("Q12", "multi_section", "Does the guide prescribe a textbook, and which assessment format is listed for Assignment 5?",
     "No prescribed textbook; a notebook and PowerPoint presentation.", [13, 14], [6, 7]),
    ("Q13", "multi_section", "How many weeks are scheduled for Bayesian Networks, and which components make up a Bayesian network?",
     "Three weeks for Unit 3; nodes (variables), edges (dependencies), and conditional probabilities.", [7, 12], [4, 5]),
    ("Q14", "multi_section", "What should an appointment email's subject contain, and what should students do if they cannot attend a class?",
     "Student number and one summarising word; email using the section 1 query template to state the case and discuss catching up.", [2, 15], [1, 8]),
    ("Q15", "unsupported", "What is the exact submission deadline for Assignment 5?", ABSTAIN, [], []),
    ("Q16", "unsupported", "What is the tuition fee for EDAB6808?", ABSTAIN, [], []),
    ("Q17", "unsupported", "What is the minimum pass mark for EDAB6808?", ABSTAIN, [], []),
    ("Q18", "unsupported", "What is the capital of France?", ABSTAIN, [], []),
], columns=["id", "type", "question", "expected_answer", "expected_printed_pages", "expected_sections"])
covered_sections = {section for values in evaluation_df["expected_sections"] for section in values if section > 0}
assert len(evaluation_df) >= 12 and evaluation_df["id"].is_unique
assert (evaluation_df["type"] == "direct").sum() >= 8
assert (evaluation_df["type"] == "multi_section").sum() >= 2
assert (evaluation_df["type"] == "unsupported").sum() >= 2
assert len(covered_sections) >= 5
print("Questions:", len(evaluation_df), "| Guide sections:", sorted(covered_sections))
print(evaluation_df["type"].value_counts().to_string())
display(evaluation_df)


Questions: 18 | Guide sections: [1, 4, 5, 6, 7, 8]
type
direct           10
multi_section     4
unsupported       4


,id,type,question,expected_answer,expected_printed_pages,expected_sections
0,Q01,direct,What is the lecturers' office number?,FGG 352,[2],[1]
1,Q02,direct,What should the email subject contain when boo...,The student number and one word summarising th...,[2],[1]
2,Q03,direct,How many credits is EDAB6808 worth?,32 credits,"[None, 5]","[0, 4]"
3,Q04,direct,What is the minimum total number of notional l...,320 hours,[5],[4]
4,Q05,direct,Is there a prescribed textbook for this module?,There is no prescribed textbook.,[13],[6]
5,Q06,direct,How much does Assignment 5 contribute to the m...,20%,[14],[7]
6,Q07,direct,How much does Assignment 6 contribute to the m...,10%,[14],[7]
7,Q08,direct,What percentage of sessions are students expec...,100%,[15],[8]
8,Q09,direct,Which Python libraries are named for data clea...,"Pandas, Matplotlib and Seaborn.",[10],[4]
9,Q10,direct,Which components make up a Bayesian network?,"Nodes (variables), edges (dependencies), and c...",[7],[4]


## 21. Verify expected sources and required facts

Source rules combine page, main section and actual fact wording. A page hit alone cannot establish correct evidence.
Hit@4 checks whether any required fact appears; All-required-evidence@4 checks every fact, including both parts of multi-section questions.
The full-page sanity check below validates ground truth before evaluating retrieval. Human review remains the final judgment of evidence, answer and citation correctness.


In [ ]:
def rule(pages, sections, pattern):
    return {"pages": pages, "sections": sections, "pattern": pattern}


source_rules = {
    "Q01": [rule([2], [1], r"\bFGG\s*352\b")],
    "Q02": [rule([2], [1], r"Student number as subject title and 1 \(one\) word to summarise the nature of the email")],
    "Q03": [rule([None, 5], [0, 4], r"\b32(?:\s+|-)(?:credit|Credits)\b")],
    "Q04": [rule([5], [4], r"\b320 notional learning hours\b")],
    "Q05": [rule([13], [6], r"no prescribed textbook")],
    "Q06": [rule([14], [7], r"Assignment\s+5\b.{0,400}?\b20%")],
    "Q07": [rule([14], [7], r"Assignment\s+6\b.{0,250}?\b10%")],
    "Q08": [rule([15], [8], r"attend\s+100%\s+of the sessions")],
    "Q09": [rule([10], [4], r"Pandas,\s*Matplotlib,\s*and Seaborn")],
    "Q10": [rule([7], [4], r"nodes \(variables\), edges \(dependencies\), and conditional probabilities")],
    "Q11": [rule([5], [4], r"minimum of 12 hours outside of class time"),
             rule([15], [8], r"attend\s+100%\s+of the sessions")],
    "Q12": [rule([13], [6], r"no prescribed textbook"),
             rule([14], [7], r"Assignment\s+5\s+Notebook\s*\+\s*PowerPoint presentation")],
    "Q13": [rule([7], [4], r"nodes \(variables\), edges \(dependencies\), and conditional probabilities"),
             rule([12], [5], r"\b3\s+3\s+This unit introduces probabilistic graphical models")],
    "Q14": [rule([2], [1], r"Student number as subject title and 1 \(one\) word to summarise the nature of the email"),
             rule([15], [8], r"If you cannot attend a class, you need to ensure that you write an email.{0,160}?query template")],
    "Q15": [], "Q16": [], "Q17": [], "Q18": [],
}
def section_numbers(chunk):
    if chunk["section"] == "Cover":
        return {0}
    return {int(n) for n in re.findall(
        r"(?:^| / )\s*(\d+)\.", chunk["section"]
    )}

def matches_source_rule(chunk, expected):
    text = " ".join(chunk["text"].split())
    return (
        chunk["printed_page"] in expected["pages"]
        and bool(section_numbers(chunk) & set(expected["sections"]))
        and re.search(expected["pattern"], text, re.I) is not None
    )

def assess_retrieval(case_id, evidence):
    rules = source_rules[case_id]
    if not rules:
        return {"hit_at_k": None, "all_evidence_at_k": None,
                "fact_ranks": []}
    ranks = [
        next((rank for rank, c in enumerate(evidence, 1)
              if matches_source_rule(c, expected)), None)
        for expected in rules
    ]
    return {
        "hit_at_k": any(rank is not None for rank in ranks),
        "all_evidence_at_k": all(rank is not None for rank in ranks),
        "fact_ranks": ranks,
    }

print("Retrieval source rules ready; answer quality is reviewed separately.")

# Verify expected facts against full source pages before scoring retrieval.
source_check_rows = []
for case_id, expected_rules in source_rules.items():
    for number, expected in enumerate(expected_rules, 1):
        matching = []
        for page in clean_pages:
            printed = page["pdf_page"] - 1 if page["pdf_page"] > 1 else None
            if printed in expected["pages"] and re.search(expected["pattern"], " ".join(page["text"].split()), re.I):
                matching.append(printed)
        assert matching, f"Expected source wording missing for {case_id}, fact {number}"
        source_check_rows.append({"ID": case_id, "Fact": number, "Verified printed pages": matching})
display(pd.DataFrame(source_check_rows))


Retrieval source rules ready; answer quality is reviewed separately.


,ID,Fact,Verified printed pages
0,Q01,1,[2]
1,Q02,1,[2]
2,Q03,1,"[None, 5]"
3,Q04,1,[5]
4,Q05,1,[13]
5,Q06,1,[14]
6,Q07,1,[14]
7,Q08,1,[15]
8,Q09,1,[10]
9,Q10,1,[7]


## 22. Compare the three retrieval methods

Warm up each method, then rotate execution order across the same 18 questions. All methods return at most four passages.
Report both required-fact coverage and retrieval time. Expansion and reranking are shared by `dense_rerank` and `hybrid`, so their comparison isolates adding lexical fusion to this design.
The default chatbot uses hybrid retrieval. These results must be observed in Colab before claiming that it improves accuracy.


In [ ]:
for mode in RETRIEVAL_MODES:
    retrieve(evaluation_df.iloc[0]["question"], mode=mode)
comparison_records = []
for number, case in enumerate(evaluation_df.to_dict("records")):
    offset = number % len(RETRIEVAL_MODES)
    order = RETRIEVAL_MODES[offset:] + RETRIEVAL_MODES[:offset]
    for mode in order:
        started = perf_counter()
        evidence = retrieve(case["question"], mode=mode)
        comparison_records.append({
            "id": case["id"], "type": case["type"], "question": case["question"],
            "mode": mode, "time_seconds": perf_counter() - started,
            "retrieved_chunks": evidence, **assess_retrieval(case["id"], evidence),
        })
comparison_summary = []
for mode in RETRIEVAL_MODES:
    records = [record for record in comparison_records if record["mode"] == mode]
    supported = [record for record in records if record["type"] != "unsupported"]
    comparison_summary.append({
        "Method": mode, "Final k": TOP_K,
        f"Hit@{TOP_K} (%)": 100 * sum(record["hit_at_k"] for record in supported) / len(supported),
        f"All-required-evidence@{TOP_K} (%)": 100 * sum(record["all_evidence_at_k"] for record in supported) / len(supported),
        "Mean retrieval seconds": sum(record["time_seconds"] for record in records) / len(records),
    })
comparison_df = pd.DataFrame(comparison_records)
display(pd.DataFrame(comparison_summary).round(3))
display(comparison_df[["id", "mode", "fact_ranks", "all_evidence_at_k", "time_seconds"]])
save_json(DATA_DIR / "retrieval_comparison.json", comparison_records)


,Method,Final k,Hit@4 (%),All-required-evidence@4 (%),Mean retrieval seconds
0,dense,4,85.714,85.714,0.023
1,dense_rerank,4,100.000,92.857,0.110
2,hybrid,4,100.000,92.857,0.107


,id,mode,fact_ranks,all_evidence_at_k,time_seconds
0,Q01,dense,[1],True,0.021039
1,Q01,dense_rerank,[1],True,0.137036
2,Q01,hybrid,[1],True,0.130410
3,Q02,dense_rerank,[1],True,0.120718
4,Q02,hybrid,[1],True,0.105352
5,Q02,dense,[1],True,0.021185
6,Q03,hybrid,[1],True,0.103284
7,Q03,dense,[1],True,0.028954
8,Q03,dense_rerank,[1],True,0.099978
9,Q04,dense,[1],True,0.022998


## 23. Inspect a retrieval comparison in full

Use any case ID to inspect cached evidence, source-fact ranks and lexical/dense/reranker diagnostics. No new model calls are made.


In [ ]:
INSPECT_ID = "Q06"
for record in comparison_records:
    if record["id"] != INSPECT_ID:
        continue
    print("\nMethod:", record["mode"], "| Fact ranks:", record["fact_ranks"])
    for rank, chunk in enumerate(record["retrieved_chunks"], 1):
        print(f"\n[{rank}] {chunk['chunk_id']} | Guide page {chunk['printed_page']} | {chunk['section']}")
        print(why_relevant(record["question"], chunk))
        print(chunk["text"])



Method: hybrid | Fact ranks: [2]

[1] edab_p15_c01 | Guide page 14 | 7. ASSESSMENT OF LEARNING
Dense rank: 7 | BM25 rank: 3 | Reranker: 1.870 | Shared words: 5, assignment, mark, module
a) The entire module mark (100%), will be based on the continuous assessment of the

    module. If a student misses an assessment opportunity without a valid reason, then the

    student will receive an incomplete for the module. A valid excuse is stipulated by the

    faculty rules, and determined by the UFS policy, not the lecturer. Therefore, rule

    implementation will be on the basis of the UFS rules, and no amount of persuasion will

    convince the lecturer of a different outcome, unless stipulated by the UFS policy. All

    people are created equal, and a fair judgement within the system will be implemented

    for all people, using evidence, and a contextual understanding of the rules and its

    limitations.

b)  If an excuse has been found valid, the student will have an opportunity

## 24. Run the full RAG evaluation

Run all 18 questions with the configured chatbot method. Record retrieved passages, selected source lines, citations, errors and end-to-end response time.
A failed model response is recorded as an error, so later cases still run. Save each completed case to Drive.
`RUN_EVALUATION = False` reloads a compatible completed run and its reviews; changed configurations are rejected.
Timings include retrieval, the evidence gate and line selection, and exclude model loading. Review expected answers only after generation.


In [ ]:
from datetime import datetime, timezone

RUN_EVALUATION = True
EVALUATION_PATH = DATA_DIR / "evaluation_results.json"
evaluation_signature = {
    "pipeline_version": PIPELINE_VERSION,
    "source_sha256": source_sha256,
    "embedding_model": EMBEDDING_MODEL,
    "generation_model": GENERATION_MODEL,
    "reranker_model": RERANKER_MODEL,
    "chunk_size": CHUNK_SIZE, "overlap": OVERLAP,
    "candidate_k": CANDIDATE_K, "bm25_k": BM25_K, "fuse_k": FUSE_K,
    "top_k": TOP_K, "rrf_k": RRF_K, "query_prefix": QUERY_PREFIX,
    "retrieval_mode": RETRIEVAL_MODE, "heading_pattern": heading_pattern, "unit_pattern": unit_pattern,
    "device": device,
    "packages": {name: version(name) for name in [
        "torch", "transformers", "sentence-transformers", "qdrant-client", "rank-bm25"
    ]},
    "cases": evaluation_df.to_dict("records"),
    "source_rules": source_rules,
}
evaluation_fingerprint = hashlib.sha256(
    json.dumps(evaluation_signature, sort_keys=True).encode()
).hexdigest()

if RUN_EVALUATION:
    if "demo" in globals():
        demo.close()
    evaluation_records = []
    quality_reviews = {}
    run_id = datetime.now(timezone.utc).isoformat()
    for case in evaluation_df.to_dict("records"):
        evidence = []
        started = perf_counter()
        try:
            evidence = retrieve(case["question"])
            result = answer_question(case["question"], evidence=evidence)
            error_message = None
        except Exception as error:
            error_message = f"{type(error).__name__}: {error}"
            result = {
                "answer": None, "sources": [],
                "retrieved_chunks": evidence,
                "verified_passages": [], "selected_line_ids": [],
                "status": "error",
            }
        elapsed = perf_counter() - started
        evaluation_records.append({
            **case, **result,
            **assess_retrieval(case["id"], evidence),
            "time_seconds": elapsed,
            "error": error_message,
        })
        save_json(EVALUATION_PATH, {
            "fingerprint": evaluation_fingerprint,
            "signature": evaluation_signature, "run_id": run_id,
            "records": evaluation_records, "reviews": quality_reviews,
        })
        print(f"{case['id']}: {result['status']} — {elapsed:.2f} seconds", flush=True)
else:
    saved = json.loads(EVALUATION_PATH.read_text(encoding="utf-8"))
    if saved["fingerprint"] != evaluation_fingerprint:
        raise ValueError("The saved evaluation uses different settings. Run a fresh evaluation.")
    evaluation_records = saved["records"]
    quality_reviews = saved.get("reviews", {})
    run_id = saved["run_id"]

assert {r["id"] for r in evaluation_records} == set(evaluation_df["id"])
evaluation_results = pd.DataFrame(evaluation_records)
with pd.option_context("display.max_colwidth", None):
    display(evaluation_results[[
        "id", "type", "answer", "status", "fact_ranks",
        "all_evidence_at_k", "time_seconds", "error",
    ]])


Q01: answered — 3.34 seconds
Q02: answered — 6.51 seconds
Q03: answered — 2.38 seconds
Q04: answered — 2.12 seconds
Q05: answered — 1.37 seconds
Q06: answered — 1.76 seconds
Q07: answered — 3.52 seconds
Q08: answered — 2.40 seconds
Q09: answered — 4.80 seconds
Q10: answered — 2.48 seconds
Q11: error — 2.64 seconds
Q12: error — 8.32 seconds
Q13: answered — 2.54 seconds
Q14: answered — 2.93 seconds
Q15: abstained — 0.43 seconds
Q16: abstained — 0.42 seconds
Q17: abstained — 0.41 seconds
Q18: abstained — 0.26 seconds


,id,type,answer,status,fact_ranks,all_evidence_at_k,time_seconds,error
0,Q01,direct,"""Office number: FGG 352"" [1]",answered,[1],True,3.343936,None
1,Q02,direct,"""hours: Student number as subject title and 1 (one) word to summarise the nature of the email. For example: 2020202011 – Inquiry Then type the"" [1]",answered,[1],True,6.507576,None
2,Q03,direct,"""32 Credits"" [1]",answered,[1],True,2.378248,None
3,Q04,direct,"""This is a 32-credit module, this means that you need to spend a minimum of 320 notional"" [1]",answered,[1],True,2.115886,None
4,Q05,direct,"""There is no prescribed textbook for this module."" [1]",answered,[1],True,1.366685,None
5,Q06,direct,"""Assignment 3 Notebook + PowerPoint presentation 30 minutes per group (20 minutes 15%"" [1]",answered,[2],True,1.763362,None
6,Q07,direct,"""Assignment 6 ePortfolio group (20 minutes presentation, 10 minutes Q&A). 10%"" [2]",answered,[2],True,3.518750,None
7,Q08,direct,"""a) The entire module mark (100%), will be based on the continuous assessment of the"" [2]",answered,[1],True,2.400787,None
8,Q09,direct,"""Matplotlib, and Seaborn) for tasks ranging from data cleaning to creating effective visualizations and performing exploratory data analysis as well as build dashboards using Looker Studio."" [1]",answered,[1],True,4.803028,None
9,Q10,direct,"""o Learn the components: nodes (variables), edges (dependencies), and conditional probabilities."" [1]",answered,[1],True,2.475545,None


## 25. Inspect answers, source evidence and citations

Read each answer against the question, expected source and actual passages. Exact quotation does not establish a complete or relevant answer.


In [ ]:
def source_location(source):
    printed = source["printed_page"]
    page = "Cover (no printed page number)" if printed is None else f"Guide page {printed}"
    ids = source.get("chunk_ids", [source["chunk_id"]])
    return (
        f"{source['source_pdf']} | {page} | PDF page {source['pdf_page']} | "
        f"{source['section']} | " + (f"{source['unit']} | " if source.get("unit") else "") + f"Chunks: {', '.join(ids)}"
    )

def inspect_evaluation(case_id):
    record = next(r for r in evaluation_records if r["id"] == case_id)
    print("Question:", record["question"])
    print("Expected answer:", record["expected_answer"])
    print("Expected printed pages:", record["expected_printed_pages"])
    print("Expected guide sections:", record["expected_sections"])
    print("Answer:", record["answer"])
    print("Error:", record["error"])
    print(f"Response time: {record['time_seconds']:.2f} seconds")
    for source in record["sources"]:
        print("\nCited", source["reference"], source_location(source))
    for n, chunk in enumerate(record["retrieved_chunks"], 1):
        print(f"\nRetrieved [{n}]", source_location(chunk))
        print(chunk["text"])

inspect_evaluation("Q02")


Question: What should the email subject contain when booking an appointment?
Expected answer: The student number and one word summarising the enquiry.
Expected printed pages: [2]
Expected guide sections: [1]
Answer: "hours: Student number as subject title and 1 (one) word to summarise the nature of the email. For example: 2020202011 – Inquiry Then type the" [1]
Error: None
Response time: 6.51 seconds

Cited [1] Study_Guide.pdf | Guide page 2 | PDF page 3 | 1. INSTRUCTOR INFORMATION | Chunks: edab_p03_c02

Retrieved [1] Study_Guide.pdf | Guide page 2 | PDF page 3 | 1. INSTRUCTOR INFORMATION | Chunks: edab_p03_c02
To book an appointment, use the template below:

 hours:          Student number as subject title and 1 (one) word to summarise the

                nature of the email. For example: 2020202011 – Inquiry Then type the

                 following message:

                Dear Dr Herkulaas/ Ms Oelofse/ Mr Trip

           My name is [your name] [your surname] [your student numbe

## 26. Define quality review and summary measures

The Evaluation review tab requires judgments of evidence relevance, answer correctness/completeness and citation support.
Unsupported cases require an answer judgment; evidence/citation correctness is not applicable.
Answer and citation accuracy stay **pending until all 18 cases are reviewed**. Expected keywords do not automatically award correctness.
Retrieval rules, refusal success, false abstentions and timings are automated diagnostics. The accuracy denominators are shown by their labels.


In [ ]:
def review_complete(record):
    review = quality_reviews.get(record["id"], {})
    if review.get("answer_correct") is None:
        return False
    return record["type"] == "unsupported" or all(
        review.get(key) is not None
        for key in ["evidence_correct", "citation_correct"]
    )

def calculate_metrics():
    supported = [r for r in evaluation_records if r["type"] != "unsupported"]
    unsupported = [r for r in evaluation_records if r["type"] == "unsupported"]
    complete = all(review_complete(r) for r in evaluation_records)
    times = pd.Series([r["time_seconds"] for r in evaluation_records])
    answer_accuracy = citation_accuracy = reviewed_evidence_accuracy = None
    if complete:
        answer_accuracy = 100 * sum(
            quality_reviews[r["id"]]["answer_correct"] for r in evaluation_records
        ) / len(evaluation_records)
        citation_accuracy = 100 * sum(
            quality_reviews[r["id"]]["citation_correct"] for r in supported
        ) / len(supported)
        reviewed_evidence_accuracy = 100 * sum(
            quality_reviews[r["id"]]["evidence_correct"] for r in supported
        ) / len(supported)
    return {
        "Questions": len(evaluation_records),
        "Quality reviews completed": sum(review_complete(r) for r in evaluation_records),
        f"Retrieval Hit@{TOP_K} (%)": 100 * sum(r["hit_at_k"] for r in supported) / len(supported),
        f"All-required-evidence@{TOP_K} (%)": 100 * sum(r["all_evidence_at_k"] for r in supported) / len(supported),
        "Reviewed evidence accuracy (%)": reviewed_evidence_accuracy,
        "Answer accuracy (%)": answer_accuracy,
        "Citation accuracy (%)": citation_accuracy,
        "Answer accuracy on supported questions (%)": (
            100 * sum(quality_reviews[r["id"]]["answer_correct"] for r in supported) / len(supported)
            if complete else None
        ),
        "False-abstention rate on supported questions (%)": 100 * sum(
            r["status"] == "abstained" for r in supported
        ) / len(supported),
        "Abstention success (%)": 100 * sum(
            r["status"] == "abstained" and r["answer"] == ABSTAIN
            and not r["sources"] and r["error"] is None
            for r in unsupported
        ) / len(unsupported),
        "Pipeline errors": sum(r["error"] is not None for r in evaluation_records),
        "Mean response seconds": float(times.mean()),
        "Median response seconds": float(times.median()),
        "95th percentile response seconds": float(times.quantile(0.95)),
    }

def metrics_markdown():
    lines = ["| Measure | Result |", "|---|---|"]
    for name, value in calculate_metrics().items():
        shown = "Pending quality review" if value is None else (
            str(value) if isinstance(value, int) else f"{value:.2f}"
        )
        lines.append(f"| {name} | {shown} |")
    return "\n".join(lines)

def save_evaluation_state():
    save_json(EVALUATION_PATH, {
        "fingerprint": evaluation_fingerprint,
        "signature": evaluation_signature, "run_id": run_id,
        "records": evaluation_records, "reviews": quality_reviews,
        "metrics": calculate_metrics(),
    })

display(Markdown(metrics_markdown()))


| Measure | Result |
|---|---|
| Questions | 18 |
| Quality reviews completed | 0 |
| Retrieval Hit@4 (%) | 100.00 |
| All-required-evidence@4 (%) | 92.86 |
| Reviewed evidence accuracy (%) | Pending quality review |
| Answer accuracy (%) | Pending quality review |
| Citation accuracy (%) | Pending quality review |
| Answer accuracy on supported questions (%) | Pending quality review |
| False-abstention rate on supported questions (%) | 0.00 |
| Abstention success (%) | 100.00 |
| Pipeline errors | 2 |
| Mean response seconds | 2.70 |
| Median response seconds | 2.44 |
| 95th percentile response seconds | 6.78 |

## 27. Optional RAGAS context metrics

Set `RUN_RAGAS = True` to calculate actual **non-LLM context recall and precision** from the saved retrieval results.
Reference contexts are exact, source-rule-matching guide spans, selected independently of the ranked results. Inspect them before interpreting scores.
These RAGAS metrics measure text similarity, not factual answer or citation correctness. They supplement the human review and require no hosted judge model.
The default skips this additional dependency; the core evaluation is complete without it. Notebook-native `await` avoids starting a second event loop.


In [ ]:
RUN_RAGAS = False
ragas_records = []
if RUN_RAGAS:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ragas"], check=True)
    from ragas import SingleTurnSample
    from ragas.metrics import NonLLMContextRecall, NonLLMContextPrecisionWithReference

    reference_candidates = [expand_context(chunk) for chunk in chunks]
    reference_contexts = {}
    for case_id, expected_rules in source_rules.items():
        selected = []
        for expected in expected_rules:
            candidates = [chunk for chunk in reference_candidates if matches_source_rule(chunk, expected)]
            if not candidates:
                raise ValueError(f"No source reference span for {case_id}; inspect chunking.")
            selected.append(min(candidates, key=lambda chunk: (len(chunk["text"]), chunk["chunk_id"]))["text"])
        reference_contexts[case_id] = list(dict.fromkeys(selected))
    display(pd.DataFrame([
        {"ID": case_id, "Reference contexts": contexts}
        for case_id, contexts in reference_contexts.items() if contexts
    ]))
    recall = NonLLMContextRecall()
    precision = NonLLMContextPrecisionWithReference()
    for record in evaluation_records:
        if record["type"] == "unsupported":
            continue
        retrieved = [chunk["text"] for chunk in record["retrieved_chunks"]]
        if not retrieved:
            ragas_records.append({"id": record["id"], "context_recall": 0.0, "context_precision": 0.0, "error": None})
            continue
        sample = SingleTurnSample(retrieved_contexts=retrieved, reference_contexts=reference_contexts[record["id"]])
        try:
            ragas_records.append({
                "id": record["id"], "context_recall": float(await recall.single_turn_ascore(sample)),
                "context_precision": float(await precision.single_turn_ascore(sample)), "error": None,
            })
        except Exception as error:
            ragas_records.append({"id": record["id"], "context_recall": None, "context_precision": None,
                                  "error": f"{type(error).__name__}: {error}"})
    display(pd.DataFrame(ragas_records))
    if any(record["error"] for record in ragas_records):
        print("RAGAS summary withheld because one or more cases failed.")
    else:
        display(pd.DataFrame(ragas_records)[["context_recall", "context_precision"]].mean().to_frame("Mean"))
    save_json(DATA_DIR / "ragas_results.json", {"fingerprint": evaluation_fingerprint, "run_id": run_id, "records": ragas_records})
else:
    print("Optional RAGAS metrics skipped. Core retrieval and quality-review measures remain available.")


Optional RAGAS metrics skipped. Core retrieval and quality-review measures remain available.


## 28. Review callbacks, chatbot replies and evidence panels

Save reviews against their evaluation run. Chat requests are serialized, and a failed question does not prevent the next question. Conversation history is displayed; each answer is grounded in fresh guide retrieval.


In [ ]:
import html
from threading import Lock

inference_lock = Lock()

def review_case(case_id):
    record = next(r for r in evaluation_records if r["id"] == case_id)
    review = quality_reviews.get(case_id, {})
    def choice(key):
        value = review.get(key)
        return "Unreviewed" if value is None else ("Correct" if value else "Incorrect")
    text = (
        f"### {case_id}: {record['question']}\n\n"
        f"**Expected answer:** {record['expected_answer']}\n\n"
        f"**Expected printed pages:** {record['expected_printed_pages']}\n\n"
        f"**Expected sections:** {record['expected_sections']}\n\n"
        f"**Source/fact retrieval ranks:** {record['fact_ranks']}\n\n"
        f"**Actual answer:**\n\n{record['answer'] or '(No answer generated)'}\n\n"
        f"**Error:** {record['error'] or 'None'}\n\n"
        f"**Time:** {record['time_seconds']:.2f} seconds\n\n"
    )
    for source in record["sources"]:
        text += f"**Cited {source['reference']}:** {source_location(source)}\n\n"
    for n, chunk in enumerate(record["retrieved_chunks"], 1):
        text += (
            f"**Retrieved [{n}]:** {source_location(chunk)}\n\n"
            f"<pre>{html.escape(chunk['text'])}</pre>\n\n"
        )
    unsupported = record["type"] == "unsupported"
    return (
        text,
        "Not applicable" if unsupported else choice("evidence_correct"),
        choice("answer_correct"),
        "Not applicable" if unsupported else choice("citation_correct"),
        review.get("notes", ""),
    )

def save_case_review(case_id, evidence_choice, answer_choice, citation_choice, notes):
    record = next(r for r in evaluation_records if r["id"] == case_id)
    required = [answer_choice] if record["type"] == "unsupported" else [
        evidence_choice, answer_choice, citation_choice
    ]
    if any(value not in {"Correct", "Incorrect"} for value in required):
        return "Choose Correct or Incorrect for each applicable judgment.", metrics_markdown()
    quality_reviews[case_id] = {
        "evidence_correct": None if record["type"] == "unsupported" else evidence_choice == "Correct",
        "answer_correct": answer_choice == "Correct",
        "citation_correct": None if record["type"] == "unsupported" else citation_choice == "Correct",
        "notes": str(notes),
    }
    save_evaluation_state()
    return f"Review saved for {case_id}.", metrics_markdown()


def evidence_markdown(question, result, elapsed):
    if result["status"] == "abstained":
        text = "**Evidence is insufficient to answer this question.**\n\n"
        evidence = result["retrieved_chunks"][:2]
        text += "Closest retrieved passages are shown for inspection; they do not establish an answer.\n\n"
    else:
        text = "**Cited source evidence**\n\n"
        evidence = result["sources"]
    for chunk in evidence:
        reference = chunk.get("reference", "Retrieved passage")
        text += f"**{reference}** — {source_location(chunk)}\n\n"
        text += f"<pre>{html.escape(chunk['text'])}</pre>\n\n"
        text += f"<details><summary>Retrieval details</summary>{html.escape(why_relevant(question, chunk))}</details>\n\n"
    return text + f"*Response time: {elapsed:.2f} seconds.*"

def chat_reply(message, history):
    started = perf_counter()
    try:
        with inference_lock:
            result = answer_question(message)
    except Exception as error:
        print(f"Chatbot error: {type(error).__name__}: {error}")
        return ("An answer could not be produced for this question. You can try another question.",
                "The answer attempt failed. Enter another question to continue.")
    reply = result["answer"]
    if result["sources"]:
        reply += "\n\n**Sources:**\n\n" + "\n\n".join(
            f"{source['reference']} {source_location(source)}" for source in result["sources"]
        )
    elapsed = perf_counter() - started
    return reply, evidence_markdown(message, result, elapsed)

def comparison_markdown(case_id):
    records = [record for record in comparison_records if record["id"] == case_id]
    parts = []
    for record in records:
        parts.append(f"### {record['mode']}\n\n**Question:** {record['question']}\n\n"
                     f"**Required-fact ranks:** {record['fact_ranks']} | **Retrieval seconds:** {record['time_seconds']:.3f}\n\n")
        for rank, chunk in enumerate(record["retrieved_chunks"], 1):
            parts.append(f"**[{rank}]** {source_location(chunk)}\n\n"
                         f"{html.escape(why_relevant(record['question'], chunk))}\n\n"
                         f"<pre>{html.escape(chunk['text'])}</pre>\n\n")
    return "\n\n---\n\n".join(parts)


## 29. Build the Gradio interface

The chatbot has a separate source-evidence panel. Two further tabs support quality review and retrieval comparison during the presentation. Clearing chat also clears its evidence panel.


In [ ]:
if "demo" in globals():
    demo.close()

initial_review = review_case("Q01")
review_options = ["Correct", "Incorrect", "Not applicable", "Unreviewed"]

with gr.Blocks(title="EDAB6808 Learner Guide Assistant") as demo:
    gr.Markdown("# EDAB6808 Learner Guide Assistant")
    gr.Markdown("Ask about the learner guide. Answers include their source evidence.")
    with gr.Tab("Chatbot"):
        evidence_panel = gr.Markdown("_Source evidence appears here after a question._", render=False)
        with gr.Row():
            with gr.Column(scale=3):
                chat_interface = gr.ChatInterface(
                    fn=chat_reply, additional_outputs=[evidence_panel],
                    examples=[
                        "What is the lecturers' office number?",
                        "How much does Assignment 5 contribute to the module mark?",
                        "Which components make up a Bayesian network?",
                        "What is the exact submission deadline for Assignment 5?",
                    ], cache_examples=False, concurrency_limit=1,
                )
            with gr.Column(scale=2):
                evidence_panel.render()
        chat_interface.chatbot.clear(
            fn=lambda: "_Source evidence appears here after a question._",
            outputs=[evidence_panel], queue=False,
        )
    with gr.Tab("Evaluation review"):
        gr.Markdown(
            "Compare the answer with the question and source. "
            "Select each applicable judgment, save it, and choose the next case."
        )
        selected_case = gr.Dropdown(
            choices=evaluation_df["id"].tolist(), value="Q01", label="Test question"
        )
        review_panel = gr.Markdown(initial_review[0])
        with gr.Row():
            evidence_rating = gr.Radio(
                review_options, value=initial_review[1], label="Correct evidence retrieved"
            )
            answer_rating = gr.Radio(
                review_options, value=initial_review[2], label="Correct and complete answer"
            )
            citation_rating = gr.Radio(
                review_options, value=initial_review[3], label="Correct supporting citations"
            )
        review_notes = gr.Textbox(value=initial_review[4], label="Review notes", lines=3)
        save_review_button = gr.Button("Save this review")
        review_status = gr.Markdown()
        review_metrics = gr.Markdown(metrics_markdown())
        selected_case.change(
            fn=review_case, inputs=[selected_case],
            outputs=[review_panel, evidence_rating, answer_rating,
                     citation_rating, review_notes],
        )
        save_review_button.click(
            fn=save_case_review,
            inputs=[selected_case, evidence_rating, answer_rating,
                    citation_rating, review_notes],
            outputs=[review_status, review_metrics],
            concurrency_limit=1,
        )

    with gr.Tab("Retrieval comparison"):
        gr.Markdown("Compare cached evidence for the same question across all three retrieval methods.")
        gr.Dataframe(value=pd.DataFrame(comparison_summary).round(3), interactive=False)
        comparison_case = gr.Dropdown(choices=evaluation_df["id"].tolist(), value="Q06", label="Comparison question")
        comparison_panel = gr.Markdown(comparison_markdown("Q06"))
        comparison_case.change(fn=comparison_markdown, inputs=[comparison_case], outputs=[comparison_panel])

demo.queue(default_concurrency_limit=1)
print("Chatbot, evidence, evaluation review and retrieval comparison interface built.")


Chatbot, evidence, evaluation review and retrieval comparison interface built.


## 30. Launch the interface in Colab

Display the app inline and create a public temporary Gradio share link. Keep the Colab runtime connected. The link includes evaluation-review controls, so share it with your demonstration audience.


In [ ]:
demo.launch(
    share=True,
    inline=True,
    prevent_thread_lock=True,
    show_error=True,
)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://76d3822d77347b6ac3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 31. Final evaluation tables and interpretation

Rerun this section after completing reviews to update the visible final tables. It reuses recorded results without generating answers again.
Compare hybrid retrieval with both baselines, report observed changes and keep a result that shows no gain when that is what occurred.
Export CSV tables for the presentation alongside the detailed JSON records; pending review remains blank in exported tables.


In [ ]:
final_rows = []
for record in evaluation_records:
    review = quality_reviews.get(record["id"], {})
    final_rows.append({
        "ID": record["id"], "Type": record["type"], "Question": record["question"],
        "Expected source pages": record["expected_printed_pages"], "Expected answer": record["expected_answer"],
        "Actual answer": record["answer"], "Status": record["status"],
        f"Source/fact Hit@{TOP_K}": record["hit_at_k"], "All required evidence": record["all_evidence_at_k"],
        "Correct evidence (review)": review.get("evidence_correct"),
        "Correct answer (review)": review.get("answer_correct"), "Correct citation (review)": review.get("citation_correct"),
        "Citation locations": " | ".join(source_location(source) for source in record["sources"]),
        "Seconds": record["time_seconds"], "Error": record["error"], "Review notes": review.get("notes", ""),
    })
final_table = pd.DataFrame(final_rows)
display(final_table.fillna("Not applicable / pending"))
display(Markdown(metrics_markdown()))
display(pd.DataFrame(comparison_summary).round(3))
key = f"All-required-evidence@{TOP_K} (%)"
hybrid = next(row for row in comparison_summary if row["Method"] == "hybrid")
for mode in ["dense", "dense_rerank"]:
    baseline = next(row for row in comparison_summary if row["Method"] == mode)
    gain = hybrid[key] - baseline[key]
    print(f"Hybrid versus {mode}: {gain:+.1f} percentage points in all-required evidence.")
    if gain <= 0:
        print("This comparison does not demonstrate a positive gain on this measure.")
metrics = calculate_metrics()
print(f"Abstention success: {metrics['Abstention success (%)']:.1f}% on four unsupported questions.")
print(f"Pipeline errors: {metrics['Pipeline errors']}.")
print(f"Mean end-to-end response time: {metrics['Mean response seconds']:.2f} seconds on {device}.")
if metrics["Answer accuracy (%)"] is None:
    print("Answer and citation accuracy remain pending the complete quality review.")
else:
    print(f"Reviewed answer accuracy: {metrics['Answer accuracy (%)']:.1f}%.")
    print(f"Reviewed citation accuracy: {metrics['Citation accuracy (%)']:.1f}%.")
save_evaluation_state()
final_table.to_csv(DATA_DIR / "evaluation_review.csv", index=False)
pd.DataFrame(comparison_summary).to_csv(DATA_DIR / "retrieval_summary.csv", index=False)
pd.DataFrame(list(metrics.items()), columns=["Measure", "Result"]).to_csv(DATA_DIR / "evaluation_summary.csv", index=False)
print("Evaluation tables saved:", DATA_DIR)


,ID,Type,Question,Expected source pages,Expected answer,Actual answer,Status,Source/fact Hit@4,All required evidence,Correct evidence (review),Correct answer (review),Correct citation (review),Citation locations,Seconds,Error,Review notes
0,Q01,direct,What is the lecturers' office number?,[2],FGG 352,"""Office number: FGG 352"" [1]",answered,True,True,Not applicable / pending,Not applicable / pending,Not applicable / pending,Study_Guide.pdf | Guide page 2 | PDF page 3 | ...,3.343936,Not applicable / pending,
1,Q02,direct,What should the email subject contain when boo...,[2],The student number and one word summarising th...,"""hours: Student number as subject title and 1 ...",answered,True,True,Not applicable / pending,Not applicable / pending,Not applicable / pending,Study_Guide.pdf | Guide page 2 | PDF page 3 | ...,6.507576,Not applicable / pending,
2,Q03,direct,How many credits is EDAB6808 worth?,"[None, 5]",32 credits,"""32 Credits"" [1]",answered,True,True,Not applicable / pending,Not applicable / pending,Not applicable / pending,Study_Guide.pdf | Cover (no printed page numbe...,2.378248,Not applicable / pending,
3,Q04,direct,What is the minimum total number of notional l...,[5],320 hours,"""This is a 32-credit module, this means that y...",answered,True,True,Not applicable / pending,Not applicable / pending,Not applicable / pending,Study_Guide.pdf | Guide page 5 | PDF page 6 | ...,2.115886,Not applicable / pending,
4,Q05,direct,Is there a prescribed textbook for this module?,[13],There is no prescribed textbook.,"""There is no prescribed textbook for this modu...",answered,True,True,Not applicable / pending,Not applicable / pending,Not applicable / pending,Study_Guide.pdf | Guide page 13 | PDF page 14 ...,1.366685,Not applicable / pending,
5,Q06,direct,How much does Assignment 5 contribute to the m...,[14],20%,"""Assignment 3 Notebook + PowerPoint presentati...",answered,True,True,Not applicable / pending,Not applicable / pending,Not applicable / pending,Study_Guide.pdf | Guide page 14 | PDF page 15 ...,1.763362,Not applicable / pending,
6,Q07,direct,How much does Assignment 6 contribute to the m...,[14],10%,"""Assignment 6 ePortfolio group (20 minutes pre...",answered,True,True,Not applicable / pending,Not applicable / pending,Not applicable / pending,Study_Guide.pdf | Guide page 14 | PDF page 15 ...,3.518750,Not applicable / pending,
7,Q08,direct,What percentage of sessions are students expec...,[15],100%,"""a) The entire module mark (100%), will be bas...",answered,True,True,Not applicable / pending,Not applicable / pending,Not applicable / pending,Study_Guide.pdf | Guide page 14 | PDF page 15 ...,2.400787,Not applicable / pending,
8,Q09,direct,Which Python libraries are named for data clea...,[10],"Pandas, Matplotlib and Seaborn.","""Matplotlib, and Seaborn) for tasks ranging fr...",answered,True,True,Not applicable / pending,Not applicable / pending,Not applicable / pending,Study_Guide.pdf | Guide page 10 | PDF page 11 ...,4.803028,Not applicable / pending,
9,Q10,direct,Which components make up a Bayesian network?,[7],"Nodes (variables), edges (dependencies), and c...","""o Learn the components: nodes (variables), ed...",answered,True,True,Not applicable / pending,Not applicable / pending,Not applicable / pending,Study_Guide.pdf | Guide page 7 | PDF page 8 | ...,2.475545,Not applicable / pending,


| Measure | Result |
|---|---|
| Questions | 18 |
| Quality reviews completed | 0 |
| Retrieval Hit@4 (%) | 100.00 |
| All-required-evidence@4 (%) | 92.86 |
| Reviewed evidence accuracy (%) | Pending quality review |
| Answer accuracy (%) | Pending quality review |
| Citation accuracy (%) | Pending quality review |
| Answer accuracy on supported questions (%) | Pending quality review |
| False-abstention rate on supported questions (%) | 0.00 |
| Abstention success (%) | 100.00 |
| Pipeline errors | 2 |
| Mean response seconds | 2.70 |
| Median response seconds | 2.44 |
| 95th percentile response seconds | 6.78 |

,Method,Final k,Hit@4 (%),All-required-evidence@4 (%),Mean retrieval seconds
0,dense,4,85.714,85.714,0.023
1,dense_rerank,4,100.000,92.857,0.110
2,hybrid,4,100.000,92.857,0.107


Hybrid versus dense: +7.1 percentage points in all-required evidence.
Hybrid versus dense_rerank: +0.0 percentage points in all-required evidence.
This comparison does not demonstrate a positive gain on this measure.
Abstention success: 100.0% on four unsupported questions.
Pipeline errors: 2.
Mean end-to-end response time: 2.70 seconds on cuda.
Answer and citation accuracy remain pending the complete quality review.
Evaluation tables saved: /content/drive/MyDrive/EDAB6808_RAG_Augmented/rag_data


## 32. Assignment coverage and limitations

| Requirement | Implementation |
|---|---|
| Guide-only extraction and preservation | Sections 3–6 |
| Justified token chunks, stable IDs, page/section/unit metadata and coverage | Sections 7–8 |
| Open-weight embeddings and vector dimension | Section 9 |
| Rerun-safe Qdrant Local and Drive persistence | Sections 10 and 19 |
| Dense retrieval plus BM25/RRF and reranking | Sections 11–13 and 22–23 |
| Local instruction-tuned Qwen, evidence gate and grounded answers | Sections 14–18 |
| Metadata citations and unsupported-question refusal | Sections 15–18 and full evaluation |
| 18 questions: 10 direct, 4 multi-section, 4 unsupported; six main sections | Sections 20–21 |
| Expected source, evidence/answer/citation review and timing | Sections 24–26 and 31 |
| Retrieval, accuracy, citation, abstention and timing measures | Sections 22, 26 and 31 |
| Optional actual RAGAS non-LLM context metrics | Section 27 |
| Repeated-question chatbot, evidence panel and review/comparison tabs | Sections 28–30 |

The source-copying guard establishes exact wording and citation mapping, not correctness or completeness. The answerability gate can still be wrong. PDF tables may be fragmented; inspect schedule and assessment answers carefully. Heading recognition uses the supplied guide's explicit structure and should be rechecked if the PDF changes.

Hybrid retrieval has more candidate-selection options, but its performance must be measured. The dense versus dense-rerank comparison includes both expansion and reranking; dense-rerank versus hybrid adds lexical fusion. Eighteen questions are still a small evaluation. The four unsupported cases do not establish refusal reliability on all possible requests.

No threshold is fitted on the evaluation questions. Reranker scores are ranking signals, not answerability probabilities. The optional RAGAS measures compare source text and supplement reviewed correctness. There is no additional LoRA training in this notebook: the supplied required-stack table calls for a local instruction-tuned Qwen/Gemma model, which is what is used here.

Report results with the actual Colab device and package versions. Model downloads require runtime internet access. Save this notebook after running and reviewing it so its measured results are visible in the submission.


## References

- Supplied EDAB6808 Learner Guide and Assignment 5 brief.
- [BGE model and query-prefix documentation](https://huggingface.co/BAAI/bge-small-en-v1.5).
- [Qwen2.5-1.5B-Instruct model card](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct).
- [rank_bm25 implementation](https://github.com/dorianbrown/rank_bm25).
- [Cross-encoder reranker](https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2).
- [Qdrant Python client](https://python-client.qdrant.tech/).
- [PyMuPDF text extraction](https://pymupdf.readthedocs.io/en/latest/recipes-text.html).
- [Gradio ChatInterface](https://www.gradio.app/docs/gradio/chatinterface).
- [Gradio sharing](https://www.gradio.app/guides/sharing-your-app).
- [RAGAS context recall](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/context_recall/).
- [RAGAS context precision](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/context_precision/).

These are methodological references, not additional chatbot evidence.
